In [29]:
import subprocess
subprocess.run(["pip", "install", "gradio", "-q"])
print("✅ Gradio ready!")

import numpy as np
import tensorflow as tf
import gradio as gr
from PIL import Image
import cv2, warnings
warnings.filterwarnings('ignore')
print(f"✅ TensorFlow {tf.__version__} | Gradio {gr.__version__}")

# ── Load Model ─────────────────────────────────────
model = tf.keras.models.load_model('/kaggle/working/digit_recognition_model.h5')
print("✅ Model loaded — Accuracy: 99.43%")

# ── Prediction Logic ───────────────────────────────
def preprocess(img_array, invert=True):
    img = cv2.resize(img_array, (28, 28))
    if invert:
        img = cv2.bitwise_not(img)
    img = img.astype('float32') / 255.0
    return img.reshape(1, 28, 28, 1)

def run_prediction(tensor):
    preds = model.predict(tensor, verbose=0)[0]
    digit = int(np.argmax(preds))
    conf  = float(np.max(preds)) * 100
    return digit, conf, preds.tolist()

def predict_upload(image):
    if image is None:
        return None, None, None
    arr = np.array(image.convert('L'))
    t   = preprocess(arr, invert=True)
    return run_prediction(t)

def predict_draw(image):
    if image is None:
        return None, None, None
    if isinstance(image, dict):
        arr = image.get('composite', None)
        if arr is None:
            layers = image.get('layers', [])
            arr = layers[0] if layers else None
        if arr is None:
            return None, None, None
    else:
        arr = image
    gray = np.array(Image.fromarray(arr.astype('uint8')).convert('L'))
    t    = preprocess(gray, invert=True)
    return run_prediction(t)

# ── CSS ────────────────────────────────────────────
CSS = """
@import url('https://fonts.googleapis.com/css2?family=JetBrains+Mono:wght@400;600;700&display=swap');

*, body, .gradio-container {
    font-family: 'JetBrains Mono', monospace !important;
    background-color: #0d1117 !important;
    color: #e6edf3 !important;
    box-sizing: border-box;
}
.gradio-container { max-width: 980px !important; margin: 0 auto !important; padding: 0 12px !important; }

.gr-group, .gr-box, .block, .panel, [data-testid="block"] {
    background: #161b22 !important;
    border: 1px solid #30363d !important;
    border-radius: 10px !important;
}

.result-card {
    background: #161b22;
    border: 1px solid #30363d;
    border-radius: 10px;
    padding: 16px;
    margin-top: 8px;
}
.digit-big       { font-size:52px; font-weight:700; color:#4cc9f0; line-height:1; font-family:'JetBrains Mono',monospace; }
.digit-big.green { color: #4ade80; }
.conf-text       { color:#fbbf24; font-size:13px; font-weight:700; text-align:right; }
.conf-label      { color:#8b949e; font-size:10px; margin-bottom:4px; }
.conf-bar-wrap   { background:#30363d; border-radius:10px; height:6px; overflow:hidden; margin-bottom:12px; }
.conf-bar-fill   { height:100%; border-radius:10px; background:linear-gradient(90deg,#4cc9f0,#4ade80); }
.probs-title     { color:#8b949e; font-size:10px; letter-spacing:1px; text-transform:uppercase; margin-bottom:6px; }
.prob-row        { display:grid; grid-template-columns:18px 1fr 46px; gap:6px; align-items:center; margin-bottom:3px; }
.prob-num        { font-size:11px; font-weight:700; color:#8b949e; text-align:center; }
.prob-num.hi     { color:#4cc9f0; }
.prob-bg         { background:#0d1117; border-radius:4px; height:5px; overflow:hidden; }
.prob-fill       { height:100%; border-radius:4px; background:#30363d; }
.prob-fill.hi    { background:linear-gradient(90deg,#4cc9f0,#2dd4bf); }
.prob-pct        { font-size:9px; color:#8b949e; text-align:right; }
.prob-pct.hi     { color:#4cc9f0; font-weight:700; }
.status-ok       { color:#4ade80; font-size:11px; text-align:center; margin-top:8px; }

.hist-wrap  { display:flex; flex-wrap:wrap; gap:8px; margin-top:10px; }
.hist-item  { background:#21262d; border:1px solid #30363d; border-radius:8px; padding:8px 14px; display:flex; align-items:center; gap:8px; }
.hist-digit { font-size:22px; font-weight:700; color:#4cc9f0; }
.hist-meta  { font-size:9px; color:#8b949e; line-height:1.5; }
.hist-empty { color:#8b949e; font-size:11px; padding:8px 0; }

.sec-lbl-yellow { color:#fbbf24 !important; font-size:11px !important; font-weight:700 !important; letter-spacing:1px; text-transform:uppercase; margin-bottom:8px; }
.sec-lbl-green  { color:#4ade80 !important; font-size:11px !important; font-weight:700 !important; letter-spacing:1px; text-transform:uppercase; margin-bottom:8px; }

#btn-predict-upload { background:#4ade80 !important; color:#0d1117 !important; font-weight:700 !important; border:none !important; border-radius:7px !important; font-size:13px !important; }
#btn-predict-draw   { background:#4ade80 !important; color:#0d1117 !important; font-weight:700 !important; border:none !important; border-radius:7px !important; }
#btn-clear          { background:#f43f5e !important; color:white   !important; font-weight:700 !important; border:none !important; border-radius:7px !important; }
#btn-save           { background:#a855f7 !important; color:white   !important; font-weight:700 !important; border:none !important; border-radius:7px !important; }

canvas { border-radius:8px !important; cursor:crosshair !important; background:white !important; }
.hide-label > label { display:none !important; }
"""

# ── HTML Builders ──────────────────────────────────
def make_result_html(digit, conf, probs, mode='upload'):
    if digit is None:
        msg = "Upload an image to see result" if mode == 'upload' else "Draw a digit and click Predict"
        return f'<div class="result-card"><p style="color:#8b949e;font-size:11px;">{msg}</p></div>'

    color  = 'green' if mode == 'draw' else ''
    label  = "RECOGNIZED DIGIT" if mode == 'upload' else "PREDICTED DIGIT"
    action = "Recognized" if mode == 'upload' else "Predicted"

    rows = ""
    for i, p in enumerate(probs):
        pct   = p * 100
        hi    = "hi" if i == digit else ""
        rows += f"""<div class="prob-row">
          <div class="prob-num {hi}">{i}</div>
          <div class="prob-bg"><div class="prob-fill {hi}" style="width:{pct:.1f}%"></div></div>
          <div class="prob-pct {hi}">{pct:.1f}%</div>
        </div>"""

    return f"""<div class="result-card">
      <div style="display:flex;align-items:center;justify-content:space-between;margin-bottom:10px;">
        <div class="digit-big {color}">{digit}</div>
        <div>
          <div style="color:#8b949e;font-size:10px;">{label}</div>
          <div class="conf-text">{conf:.2f}%</div>
        </div>
      </div>
      <div class="conf-label">Confidence Score</div>
      <div class="conf-bar-wrap"><div class="conf-bar-fill" style="width:{conf:.1f}%"></div></div>
      <div class="probs-title">ALL DIGIT PROBABILITIES</div>
      {rows}
      <div class="status-ok">✅ {action} digit: {digit} ({conf:.2f}% confidence)</div>
    </div>"""

history = []

def make_history_html():
    if not history:
        return '<div class="hist-wrap"><span class="hist-empty">No predictions yet — upload or draw a digit!</span></div>'
    return '<div class="hist-wrap">' + ''.join(
        f'<div class="hist-item"><div class="hist-digit">{h["digit"]}</div>'
        f'<div class="hist-meta">{h["icon"]} {h["conf"]:.1f}%<br>🕐 {h["time"]}</div></div>'
        for h in history
    ) + '</div>'

def add_history(digit, conf, icon):
    from datetime import datetime
    history.insert(0, {'digit': digit, 'conf': conf, 'icon': icon,
                        'time': datetime.now().strftime('%I:%M %p')})
    if len(history) > 8:
        history.pop()

# ── Build App ──────────────────────────────────────
with gr.Blocks(css=CSS, title="Handwritten Digit Recognition — Khansa Bint-e-Zia") as demo:

    gr.HTML("""
    <div style="text-align:center;padding:22px 0 14px;">
      <h1 style="color:#4cc9f0;font-size:24px;font-weight:700;">✍️ Handwritten Digit Recognition</h1>
      <p style="color:#8b949e;font-size:11px;margin-top:5px;">CNN Model trained on MNIST Dataset — by Khansa Bint-e-Zia</p>
      <div style="display:flex;gap:8px;justify-content:center;flex-wrap:wrap;margin-top:10px;">
        <span style="border:1px solid #4cc9f0;color:#4cc9f0;border-radius:20px;padding:3px 12px;font-size:10px;">Python</span>
        <span style="border:1px solid #4cc9f0;color:#4cc9f0;border-radius:20px;padding:3px 12px;font-size:10px;">TensorFlow</span>
        <span style="border:1px solid #4ade80;color:#4ade80;border-radius:20px;padding:3px 12px;font-size:10px;">CNN</span>
        <span style="border:1px solid #4ade80;color:#4ade80;border-radius:20px;padding:3px 12px;font-size:10px;">MNIST</span>
        <span style="border:1px solid #a855f7;color:#a855f7;border-radius:20px;padding:3px 12px;font-size:10px;">AI Project</span>
      </div>
    </div>""")

    with gr.Row(equal_height=False):

        # LEFT — Upload
        with gr.Column(scale=1):
            gr.HTML('<p class="sec-lbl-yellow">📂 &nbsp;UPLOAD IMAGE</p>')
            upload_img = gr.Image(type="pil", label="", sources=["upload"],
                                  height=220, elem_classes=["hide-label"])
            btn_upload = gr.Button("🔍  Predict Uploaded Image", elem_id="btn-predict-upload")
            upload_out = gr.HTML(make_result_html(None, None, None, 'upload'))

        # RIGHT — Draw
        with gr.Column(scale=1):
            gr.HTML('<p class="sec-lbl-green">✏️ &nbsp;DRAW A DIGIT</p>')
            draw_pad = gr.Sketchpad(
                label="", type="numpy", height=220,
                canvas_size=(400, 220),
                brush=gr.Brush(default_size=16, colors=["#000000"], default_color="#000000"),
                elem_classes=["hide-label"]
            )
            gr.HTML('<p style="color:#8b949e;font-size:10px;text-align:center;margin:4px 0 6px;">Draw any digit (0–9) using your mouse or finger</p>')
            with gr.Row():
                btn_predict = gr.Button("🔍 Predict", elem_id="btn-predict-draw")
                btn_clear   = gr.Button("🗑️ Clear",   elem_id="btn-clear")
                btn_save    = gr.Button("💾 Save",    elem_id="btn-save")
            draw_out = gr.HTML(make_result_html(None, None, None, 'draw'))

    # History
    gr.HTML('<p class="sec-lbl-yellow" style="margin-top:16px;">📋 &nbsp;PREDICTION HISTORY</p>')
    history_out = gr.HTML(make_history_html())

    gr.HTML("""
    <div style="text-align:center;color:#8b949e;font-size:10px;padding:18px 0 6px;">
      Khansa Bint-e-Zia &nbsp;|&nbsp;
      <a href="https://github.com/Khansa972" target="_blank" style="color:#4cc9f0;text-decoration:none;">github.com/Khansa972</a>
      &nbsp;|&nbsp; CNN · MNIST · TensorFlow
    </div>""")

    # Actions
    def on_upload(img):
        digit, conf, probs = predict_upload(img)
        if digit is None:
            return make_result_html(None,None,None,'upload'), make_history_html()
        add_history(digit, conf, '📂')
        return make_result_html(digit, conf, probs, 'upload'), make_history_html()

    def on_draw(img):
        digit, conf, probs = predict_draw(img)
        if digit is None:
            return make_result_html(None,None,None,'draw'), make_history_html()
        add_history(digit, conf, '✏️')
        return make_result_html(digit, conf, probs, 'draw'), make_history_html()

    def on_clear():
        return None, make_result_html(None,None,None,'draw')

    def on_save(img):
        if img is None: return
        arr = img.get('composite') if isinstance(img, dict) else img
        if arr is not None:
            Image.fromarray(arr.astype('uint8')).save('/kaggle/working/my_drawing.png')
            print("✅ Saved: /kaggle/working/my_drawing.png")

    btn_upload.click(fn=on_upload,  inputs=[upload_img], outputs=[upload_out, history_out])
    btn_predict.click(fn=on_draw,   inputs=[draw_pad],   outputs=[draw_out,   history_out])
    btn_clear.click(fn=on_clear,    inputs=[],           outputs=[draw_pad,   draw_out])
    btn_save.click(fn=on_save,      inputs=[draw_pad],   outputs=[])

print("\n" + "="*55)
print("🚀 Launching — click the public URL below!")
print("="*55)
demo.launch(share=True, debug=False)

✅ Gradio ready!
✅ TensorFlow 2.19.0 | Gradio 5.50.0
✅ Model loaded — Accuracy: 99.43%

🚀 Launching — click the public URL below!
* Running on local URL:  http://127.0.0.1:7864
* Running on public URL: https://aa31fbaafc219aeef6.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
